In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.preprocessing import RobustScaler

In [69]:
df=pd.read_csv('Titanic-Dataset.csv')

In [70]:
df['Family']=df['Parch']+df['SibSp']+1

In [72]:
columns_to_drop=['PassengerId','Name','Ticket','Cabin','Parch','SibSp']

In [73]:
new_df=df.drop(columns=columns_to_drop)

In [3]:
class IQROutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds_ = {}
        self.upper_bounds_ = {}

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        
        q25 = X_df.quantile(0.25)
        q75 = X_df.quantile(0.75)
        iqr = q75 - q25
        
        for col in X_df.columns:
            self.lower_bounds_[col] = q25[col] - (self.factor * iqr[col])
            self.upper_bounds_[col] = q75[col] + (self.factor * iqr[col])
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        
        for col in X_df.columns:
            if col in self.lower_bounds_:
                X_df[col] = np.clip(X_df[col], self.lower_bounds_[col], self.upper_bounds_[col])
        
        return X_df.to_numpy() if isinstance(X, np.ndarray) else X_df

In [75]:
x_train,x_test,y_train,y_test=train_test_split(new_df.drop(columns=['Survived']),new_df['Survived'],random_state=42,test_size=0.25)

In [6]:
ct1=ColumnTransformer([
    ('impute_age',SimpleImputer(strategy='median'),[2]),
    ('impute_embarked',SimpleImputer(strategy='most_frequent'),[4])
    
],remainder='passthrough')

In [7]:
ct2=ColumnTransformer([
    ('OHE_sex',OneHotEncoder(dtype=np.int32),[3]),
    ('OHE_embarked',OneHotEncoder(dtype=np.int32),[1])
],remainder='passthrough')

In [8]:
ct3=ColumnTransformer([
    ('Capping_outlier',IQROutlierCapper(factor=1.5),[5,7])
],remainder='passthrough')

In [9]:
pipe=Pipeline([
    ('ct1',ct1),
    ('ct2',ct2)
])

In [146]:
x_train_transform = pipe.fit_transform(x_train)

In [155]:
x_test_transform = pipe.transform(x_test)